# Notebook 2 — Feature Engineering
**Port Intelligence BI Training Program**

This notebook walks through the 46-feature pipeline used to train the three port-intelligence
models. You will learn how each feature group is constructed, inspect a single vessel's
inference vector, explore feature-target correlations, and understand the role of cyclical
encoding and rolling-window aggregates.

---

## 1  Setup
Load libraries and import the feature-engineering module from the project root.

In [ ]:
import sys
sys.path.insert(0, '../..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from features import build_features, ALL_FEATURES
from api.predictor import build_inference_features

sns.set_theme(style='whitegrid', palette='tab10', font_scale=1.05)
plt.rcParams['figure.dpi'] = 110

# Load raw data
raw = pd.read_parquet('../../data/port_calls.parquet')
for col in ['eta_planned', 'ata_actual', 'atb', 'etd', 'atd_actual', 'created_date']:
    if col in raw.columns:
        raw[col] = pd.to_datetime(raw[col])

print(f'Raw dataset: {raw.shape[0]:,} rows x {raw.shape[1]} columns')
print(f'ALL_FEATURES list has {len(ALL_FEATURES)} entries.')

## 2  Run `build_features` on the Full Dataset
`build_features(df)` sorts by `ata_actual`, computes rolling window counts,
adds cyclical encodings, and encodes categoricals. It returns the full augmented DataFrame.

In [ ]:
print('Running build_features (may take ~30 s due to rolling window calculations) ...')
feat_df = build_features(raw)

print(f'\nAugmented DataFrame: {feat_df.shape[0]:,} rows x {feat_df.shape[1]} columns')
print(f'New columns added: {feat_df.shape[1] - raw.shape[1]}')

print('\nAll engineered columns (from ALL_FEATURES):')
for i, f in enumerate(ALL_FEATURES):
    print(f'  {i+1:>2}. {f}')

## 3  Feature Category Breakdown
The 46 features fall into five logical groups:
temporal, vessel, port/operational, weather, and cargo.

In [ ]:
feature_groups = {
    'Temporal': [
        'hour_of_day', 'day_of_week', 'day_of_month', 'month', 'week_of_year',
        'quarter', 'is_weekend', 'is_peak_hour', 'holiday_flag',
        'days_since_holiday', 'eta_deviation_min',
        'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos',
    ],
    'Vessel': [
        'company_tier', 'is_container', 'teu_cap_norm', 'dwt_norm', 'loa_norm',
        'load_factor', 'dwt',
        'vessel_type_enc', 'vessel_teu_class_enc',
    ],
    'Operational / Queue': [
        'port_haifa', 'berth_num', 'arrivals_6h', 'arrivals_12h', 'arrivals_24h',
        'berth_competition_ratio', 'queue_position', 'crane_sharing_risk',
        'service_frequency', 'cranes_used',
        'port_name_enc', 'berth_zone_enc', 'service_line_enc',
    ],
    'Weather': [
        'weather_wind_knots', 'weather_storm_flag', 'berth_competition',
    ],
    'Cargo': [
        'cargo_tons_log', 'teu_imbalance',
    ],
}

print(f'{"Group":<25} {"Count":>6}   Features')
print('-' * 80)
total = 0
for group, features in feature_groups.items():
    print(f'{group:<25} {len(features):>6}   {features[:4]}...' if len(features) > 4
          else f'{group:<25} {len(features):>6}   {features}')
    total += len(features)
print('-' * 80)
print(f'{"TOTAL (with overlaps)":<25} {total:>6}')
print(f'{"ALL_FEATURES (deduplicated)":<25} {len(ALL_FEATURES):>6}')

# Visualise as bar chart
fig, ax = plt.subplots(figsize=(8, 3))
group_names = list(feature_groups.keys())
group_sizes = [len(v) for v in feature_groups.values()]
bars = ax.barh(group_names, group_sizes,
               color=sns.color_palette('tab10', len(group_names)))
for bar, size in zip(bars, group_sizes):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(size), va='center')
ax.set_xlabel('Number of features')
ax.set_title('Feature Count by Category')
plt.tight_layout()
plt.show()

## 4  Manual Inference Feature Vector
At prediction time we don't have a historical DataFrame — we receive a single
vessel request. `build_inference_features` constructs the same 46-value vector
without needing rolling-window history (context signals are passed explicitly).

In [ ]:
from datetime import datetime

# Example vessel: MSC CHIARA, CONTAINER, calling Haifa on a Tuesday morning
fvec = build_inference_features(
    eta_planned       = datetime(2025, 6, 10, 7, 0),
    ata_actual        = datetime(2025, 6, 10, 9, 30),   # arrived 2.5h late
    port_name         = 'Haifa',
    vessel_type       = 'CONTAINER',
    teu_capacity      = 14_500,
    dwt               = 160_000,
    loa               = 366,
    company_name      = 'MSC',
    service_line      = 'Asia-EU',
    berth_id          = 'H7',
    cranes_used       = 4,
    cargo_tons        = 45_000,
    teu_loaded        = 6_200,
    teu_discharged    = 5_800,
    weather_wind_knots= 12.5,
    berth_competition = 1.8,
    arrivals_6h       = 8,
    arrivals_12h      = 15,
    arrivals_24h      = 28,
    queue_position    = 7,
)

print(f'Inference vector shape: {fvec.shape}')
print(f'\n{"#":>3}  {"Feature Name":<30}  {"Value":>10}')
print('-' * 50)
for i, (name, val) in enumerate(zip(ALL_FEATURES, fvec)):
    print(f'{i+1:>3}. {name:<30}  {val:>10.4f}')

## 5  Correlation Heatmap
Which engineered features correlate most strongly with the waiting-time target?

In [ ]:
target = 'waiting_anchor_hours'

# Compute Pearson correlations between each feature and the target
available = [f for f in ALL_FEATURES if f in feat_df.columns]
corr_series = (
    feat_df[available + [target]]
    .corr()[target]
    .drop(target)
    .sort_values(key=abs, ascending=False)
)

top15 = corr_series.head(15)

print('Top 15 features by |Pearson r| with waiting_anchor_hours:')
for feat, r in top15.items():
    bar = '#' * int(abs(r) * 40)
    print(f'  {feat:<35} r={r:+.4f}  {bar}')

# Heatmap of top-15 inter-feature correlations
corr_matrix = feat_df[list(top15.index)].corr()

fig, ax = plt.subplots(figsize=(11, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(corr_matrix, ax=ax, cmap='RdBu_r', vmin=-1, vmax=1,
            annot=True, fmt='.2f', linewidths=0.4,
            annot_kws={'fontsize': 8})
ax.set_title('Correlation Matrix — Top 15 Features (by |r| with waiting time)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

## 6  Cyclical Encoding Demo
Hour-of-day is a circular variable: hour 23 is close to hour 0.
A raw integer encoding breaks this continuity. Sin/cos encoding fixes it.

In [ ]:
hours = np.arange(0, 24)
hour_sin = np.sin(2 * np.pi * hours / 24)
hour_cos = np.cos(2 * np.pi * hours / 24)

fig = plt.figure(figsize=(14, 5))
gs = gridspec.GridSpec(1, 2, figure=fig)

# Left: raw vs sin/cos as line plots
ax0 = fig.add_subplot(gs[0, 0])
ax0.plot(hours, hours / 23.0, label='Raw (normalised)', color='gray',
         linewidth=2, linestyle='--')
ax0.plot(hours, hour_sin, label='sin(hour)', color='#1f77b4', linewidth=2)
ax0.plot(hours, hour_cos, label='cos(hour)', color='#ff7f0e', linewidth=2)
ax0.set_xlabel('Hour of day')
ax0.set_ylabel('Encoded value')
ax0.set_title('Raw vs Cyclical Encoding — Hour of Day')
ax0.set_xticks(range(0, 24, 2))
ax0.legend()

# Right: circle plot
ax1 = fig.add_subplot(gs[0, 1], projection='polar')
theta = 2 * np.pi * hours / 24
sc = ax1.scatter(theta, np.ones_like(hours), c=hours,
                 cmap='hsv', s=120, zorder=5)
for h in hours:
    ax1.text(2 * np.pi * h / 24, 1.18, str(h),
             ha='center', va='center', fontsize=7.5)
ax1.set_rticks([])
ax1.set_title('Hours on the Unit Circle\n(colour = hour)', pad=20)
fig.colorbar(sc, ax=ax1, shrink=0.6, label='Hour')

plt.tight_layout()
plt.show()

print('Key insight: hours 23 and 0 are adjacent on the circle')
print(f'  Raw distance   23 vs  0 : {abs(23 - 0):>5}')
print(f'  Euclidean dist sin/cos  : {np.sqrt((hour_sin[23]-hour_sin[0])**2 + (hour_cos[23]-hour_cos[0])**2):.4f}')
print(f'  Euclidean dist sin/cos 11vs12: {np.sqrt((hour_sin[11]-hour_sin[12])**2 + (hour_cos[11]-hour_cos[12])**2):.4f}')

## 7  Rolling Arrivals Feature
`arrivals_6h`, `arrivals_12h`, and `arrivals_24h` count how many vessels
arrived at the same port in the trailing window. They capture congestion buildup.

In [ ]:
# Pick a busy sample day at Haifa
sample_date = '2025-03-15'
day_mask = (
    (feat_df['ata_actual'].dt.date == pd.Timestamp(sample_date).date()) &
    (feat_df['port_name'] == 'Haifa')
)
day_df = feat_df.loc[day_mask].sort_values('ata_actual').reset_index(drop=True)

if len(day_df) == 0:
    # Fallback: use the busiest Haifa day in the dataset
    haifa = feat_df[feat_df['port_name'] == 'Haifa'].copy()
    haifa['date_only'] = haifa['ata_actual'].dt.date
    sample_date = str(haifa.groupby('date_only').size().idxmax())
    day_mask = haifa['date_only'] == pd.Timestamp(sample_date).date()
    day_df = haifa.loc[day_mask].sort_values('ata_actual').reset_index(drop=True)

print(f'Sample day: {sample_date}  |  Haifa arrivals that day: {len(day_df)}')

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Top: arrival events as a stem plot
axes[0].stem(day_df.index, np.ones(len(day_df)),
             linefmt='C0-', markerfmt='C0o', basefmt='k-')
axes[0].set_ylabel('Arrival event')
axes[0].set_title(f'Haifa Arrivals on {sample_date} and Trailing Counts')
axes[0].yaxis.set_visible(False)

# Bottom: rolling arrival counts
for col, color, label in [
        ('arrivals_6h',  '#1f77b4', '6-hour window'),
        ('arrivals_12h', '#ff7f0e', '12-hour window'),
        ('arrivals_24h', '#2ca02c', '24-hour window')]:
    if col in day_df.columns:
        axes[1].plot(day_df.index, day_df[col], marker='o', markersize=4,
                     linewidth=1.8, label=label, color=color)

axes[1].set_xlabel('Arrival sequence (sorted by time)')
axes[1].set_ylabel('Trailing arrivals count')
axes[1].legend()

# Annotate with hour labels
hour_ticks = [i for i in day_df.index if i % max(1, len(day_df)//8) == 0]
axes[1].set_xticks(hour_ticks)
axes[1].set_xticklabels(
    [day_df['ata_actual'].iloc[i].strftime('%H:%M') for i in hour_ticks],
    rotation=30
)

plt.tight_layout()
plt.show()

print('Correlation of rolling arrivals with waiting time (full dataset):')
for col in ['arrivals_6h', 'arrivals_12h', 'arrivals_24h']:
    if col in feat_df.columns:
        r = feat_df[col].corr(feat_df['waiting_anchor_hours'])
        print(f'  {col:<20} r = {r:.4f}')

## 8  Exercises

**Exercise 1** — Add a new feature `log_wait` = log1p of `waiting_anchor_hours`
to `feat_df`, then compute its Pearson correlation with `berth_competition`.

**Exercise 2** — Construct an inference vector for a BULK vessel arriving at Ashdod
on a Friday at 22:00 with wind speed 30 knots. Print all 46 values.

**Exercise 3** — Plot the sin/cos encoding for `day_of_week` (0–6) on a polar chart,
labelling each point with the day name.

In [ ]:
# Exercise 1
# Add log_wait column and compute correlation with berth_competition

feat_df['log_wait'] = np.log1p(feat_df['???'])   # YOUR CODE HERE

r = feat_df['log_wait'].corr(feat_df['???'])      # YOUR CODE HERE
print(f'Pearson r(log_wait, berth_competition) = {r:.4f}')

In [ ]:
# Exercise 2
# Build inference vector for a BULK vessel at Ashdod, Friday 22:00, wind 30 kn

fvec2 = build_inference_features(
    eta_planned        = datetime(2025, 9, 5, 20, 0),
    ata_actual         = datetime(2025, 9, 5, 22, 0),
    port_name          = '???',          # YOUR CODE HERE
    vessel_type        = '???',          # YOUR CODE HERE
    teu_capacity       = 0,
    dwt                = 75_000,
    loa                = 225,
    company_name       = 'Other',
    service_line       = 'West-Africa',
    berth_id           = 'A3',
    cranes_used        = 2,
    cargo_tons         = 55_000,
    teu_loaded         = 0,
    teu_discharged     = 0,
    weather_wind_knots = ???,            # YOUR CODE HERE
    berth_competition  = 2.1,
)
print('BULK / Ashdod inference vector:')
for name, val in zip(ALL_FEATURES, fvec2):
    print(f'  {name:<30} {val:.4f}')

In [ ]:
# Exercise 3
# Polar chart of sin/cos encoding for day_of_week (0=Mon, 6=Sun)

days = np.arange(0, 7)
day_names = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']
theta = 2 * np.pi * days / 7

fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={'projection': 'polar'})
ax.scatter(theta, np.ones_like(days), s=150,
           c=days, cmap='tab10', zorder=5)

for d, t, name in zip(days, theta, day_names):
    ax.text(t, ???, name, ha='center', va='center', fontsize=9)  # YOUR CODE HERE: choose radius

ax.set_rticks([])
ax.set_title('Day-of-Week Cyclical Encoding\n(sin/cos on unit circle)')
plt.tight_layout()
plt.show()